# Objectives
Welcome to the third Natural Language Processing lab!

Today's objectives are to:

1. 🔢 Explore different **text representations** as vectors of token weights
2. 📐 Calculate **cosine similarities** between documents.
3. 🔎 Build a **search engine** to compare a user query with corpus documents.

# Notebook Setup

In [2]:
#| code-summary: 'Import packages'
#| output: false
# Data packages
import pandas as pd
pd.set_option('display.max_colwidth', None) #default: 50 chars
pd.set_option('display.max_columns', None) #default: 20 columns
import numpy as np

# NLP packages
import spacy
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pymupdf

# Plotting package
import plotly.express as px

# Utility libraries
from pathlib import Path
from itertools import chain
from collections import Counter

## Toy Corpus
The demonstration corpus comes from the textbook (Lane & Dyshel, 2024) GitHub's [bias_intro.txt](https://gitlab.com/tangibleai/nlpia2/-/raw/main/src/nlpia2/ch03/bias_intro.txt) and consists of a subset of text from the Wikipedia [page](https://en.wikipedia.org/wiki/Algorithmic_bias) for "Algorithmic bias". 

In [3]:
#| code-summary: "Setup toy corpus from Wikipedia"
#| code-fold: false
docs = ['Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.',
        'Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.',
        'Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.',
        'The study of algorithmic bias is most concerned with algorithms that reflect "systematic and unfair" discrimination.'
       ]

## SpaCy Pipeline
We setup the spaCy [pipeline](https://spacy.io/usage/processing-pipelines) similarly to Lab 2 and apply it to the toy corpus.

In [5]:
#| code-summary: "Setup spaCy tokenization and normalization"
#| code-fold: false

# English language model
spacy.cli.download('en_core_web_sm') #Download (once)
nlp = spacy.load('en_core_web_sm')

# Callable function used as the custom tokenizer for scikit-learn's *Vectorizer class
def spacy_tokenizer(text):
    '''Returns list of tokens'''
    doc = nlp(text) # Tokenization pipeline
    # MODIFY BELOW: lower_, is_alpha, lemma_, is_stop (normalization from Lab 2)
    tokens = [token.lower_ for token in doc if token.is_alpha]
    return tokens

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
#| code-summary: "Apply spaCy pipeline first for some downstream vectorizations and functions"
docs_as_tokens = [spacy_tokenizer(doc) for doc in docs]
for doc in docs_as_tokens:
    print(doc)

['algorithmic', 'bias', 'describes', 'systematic', 'and', 'repeatable', 'errors', 'in', 'a', 'computer', 'system', 'that', 'create', 'unfair', 'outcomes', 'such', 'as', 'privileging', 'one', 'arbitrary', 'group', 'of', 'users', 'over', 'others']
['bias', 'can', 'emerge', 'due', 'to', 'many', 'factors', 'including', 'but', 'not', 'limited', 'to', 'the', 'design', 'of', 'the', 'algorithm', 'or', 'the', 'unintended', 'or', 'unanticipated', 'use', 'or', 'decisions', 'relating', 'to', 'the', 'way', 'data', 'is', 'coded', 'collected', 'selected', 'or', 'used', 'to', 'train', 'the', 'algorithm']
['algorithmic', 'bias', 'is', 'found', 'across', 'platforms', 'including', 'but', 'not', 'limited', 'to', 'search', 'engine', 'results', 'and', 'social', 'media', 'platforms', 'and', 'can', 'have', 'impacts', 'ranging', 'from', 'inadvertent', 'privacy', 'violations', 'to', 'reinforcing', 'social', 'biases', 'of', 'race', 'gender', 'sexuality', 'and', 'ethnicity']
['the', 'study', 'of', 'algorithmic', 

## Helper Function

In [7]:
#| code-summary: "Define function to print sparsity information" 
def print_sparsity(X_csr):
    '''Calculates and prints sparsity %'''
    sparsity = 1 - (X_csr.nnz / (X_csr.shape[0] * X_csr.shape[1]))
    print(f'{repr(X_csr)} ... and sparsity {sparsity:.2%}.')

# One-Hot Encoding

**One-hot encoding** preserves every token position but requires the most empty space (super sparse!) as EACH document needs a position-term matrix.

Scikit-learn's [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) class is not specifically designed for text data so we need to:
- Manually create the corpus vocabulary as features for `.fit()`
- Reshape the docs as token columns: `[[t] for t in doc_as_tokens]`
- Apply the encoding <span style="color:red">PER-document</span>

In [8]:
#| code-summary: "Run steps for using a one-hot encoder to vectorize the toy corpus"
# 1. Configure encoder
onehot_encoder = OneHotEncoder(handle_unknown='ignore')

# 2. Fit to corpus vocabulary
corpus_vocab = sorted(set(t for doc in docs_as_tokens for t in doc)) # all unique tokens
print(f'The vocabulary size is {len(corpus_vocab)} tokens.\n')
onehot_encoder.fit([[t] for t in corpus_vocab])

# 3. Transform (apply encoding) PER-document
def ohe_transform_to_df(encoder, doc_as_tokens):
    '''Transforms OHE matrix and displays as a DataFrame'''
    X_ohe = encoder.transform([[t] for t in doc_as_tokens])
    print_sparsity(X_ohe)
    df_ohe = pd.DataFrame(X_ohe.toarray(), index=doc_as_tokens, columns=encoder.categories_)
    display(df_ohe)

The vocabulary size is 80 tokens.



In [9]:
#| code-summary: "MODIFY document index number to view PER-document matrices"
ohe_transform_to_df(onehot_encoder, docs_as_tokens[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 25 stored elements and shape (25, 80)> ... and sparsity 98.75%.


,a,across,algorithm,algorithmic,algorithms,and,arbitrary,as,bias,biases,but,can,coded,collected,computer,concerned,create,data,decisions,describes,design,discrimination,due,emerge,engine,errors,ethnicity,factors,found,from,gender,group,have,impacts,in,inadvertent,including,is,limited,many,media,most,not,of,one,or,others,outcomes,over,platforms,privacy,privileging,race,ranging,reflect,reinforcing,relating,repeatable,results,search,selected,sexuality,social,study,such,system,systematic,that,the,to,train,unanticipated,unfair,unintended,use,used,users,violations,way,with
algorithmic,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
bias,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
describes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
systematic,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
and,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
repeatable,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
errors,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
in,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
a,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
computer,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Bag of Words Vectorization
## CountVectorizer
Scikit-learn's [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) class vectorizes lists of documents (corpus) into a "Bag of Words" **vector for each document** resulting in (one type of) **document-term matrix** to represent the entire corpus. The values are simply the count of word occurrences, without any position information, thus the name of "Bag" 💰 of Words 🔤.

Steps for [usage](https://scikit-learn.org/stable/modules/feature_extraction.html#common-vectorizer-usage) are:

1. Instantiate `CountVectorizer` and configure options
2. `fit` to corpus vocabulary (can be combined with transform)
3. `transform` corpus of documents into a sparse **document-term matrix**

## Tokenization and Normalization (vs. spaCy)
CountVectorizer can also wrap SOME tokenization and normalization steps that overlap with **spaCy**:

- Tokenization ⚠️
    - Regex: 2+ character alphanumeric word boundaries (default) (Option A)
    - OR use **spaCy** linguistic tokenizer (Options B/C)
- Case folding ✅
    - `lowercase` (default)
- Lemmatization ❌
    - use **spaCy** `token.lemma_` (Options B/C)
- Stop words ⚠️
    - Optional `stop_words` using provided `'english'` or custom list
    - OR use **spaCy** `token.is_stop` (Options B/C)
    - ... but recall [warnings](https://scikit-learn.org/stable/modules/feature_extraction.html#stop-words)

In [10]:
#| code-summary: "Option A: Internally use default CountVectorizer tokenization and normalization"
#| code-fold: false
bow_vectorizer_a = CountVectorizer(
    token_pattern = r"(?u)\b\w\w+\b" # default pattern = r"(?u)\b\w\w+\b"
)
bow_vectorizer_a.fit(docs)
print(f'The vocabulary size is {len(bow_vectorizer_a.vocabulary_)} tokens.')
bow_vectorizer_a.get_feature_names_out()

The vocabulary size is 79 tokens.


array(['across', 'algorithm', 'algorithmic', 'algorithms', 'and',
       'arbitrary', 'as', 'bias', 'biases', 'but', 'can', 'coded',
       'collected', 'computer', 'concerned', 'create', 'data',
       'decisions', 'describes', 'design', 'discrimination', 'due',
       'emerge', 'engine', 'errors', 'ethnicity', 'factors', 'found',
       'from', 'gender', 'group', 'have', 'impacts', 'in', 'inadvertent',
       'including', 'is', 'limited', 'many', 'media', 'most', 'not', 'of',
       'one', 'or', 'others', 'outcomes', 'over', 'platforms', 'privacy',
       'privileging', 'race', 'ranging', 'reflect', 'reinforcing',
       'relating', 'repeatable', 'results', 'search', 'selected',
       'sexuality', 'social', 'study', 'such', 'system', 'systematic',
       'that', 'the', 'to', 'train', 'unanticipated', 'unfair',
       'unintended', 'use', 'used', 'users', 'violations', 'way', 'with'],
      dtype=object)

In [11]:
#| code-summary: "Option B: Internally use spaCy (or other custom) tokenizer/normalizer"
#| code-fold: false
bow_vectorizer_b = CountVectorizer(
    tokenizer = spacy_tokenizer # function defined above
)
bow_vectorizer_b.fit(docs)
print(f'The vocabulary size is {len(bow_vectorizer_b.vocabulary_)} tokens.')
bow_vectorizer_b.get_feature_names_out()

The vocabulary size is 80 tokens.


c:\dev\hertie\Sem 5\NLP\sem5-nlp-labs\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


array(['a', 'across', 'algorithm', 'algorithmic', 'algorithms', 'and',
       'arbitrary', 'as', 'bias', 'biases', 'but', 'can', 'coded',
       'collected', 'computer', 'concerned', 'create', 'data',
       'decisions', 'describes', 'design', 'discrimination', 'due',
       'emerge', 'engine', 'errors', 'ethnicity', 'factors', 'found',
       'from', 'gender', 'group', 'have', 'impacts', 'in', 'inadvertent',
       'including', 'is', 'limited', 'many', 'media', 'most', 'not', 'of',
       'one', 'or', 'others', 'outcomes', 'over', 'platforms', 'privacy',
       'privileging', 'race', 'ranging', 'reflect', 'reinforcing',
       'relating', 'repeatable', 'results', 'search', 'selected',
       'sexuality', 'social', 'study', 'such', 'system', 'systematic',
       'that', 'the', 'to', 'train', 'unanticipated', 'unfair',
       'unintended', 'use', 'used', 'users', 'violations', 'way', 'with'],
      dtype=object)

In [12]:
#| code-summary: "Option C: Override internal tokenization, expect pre-tokenized/normalized input (Lab 2)"
#| code-fold: false
bow_vectorizer_c = CountVectorizer(
    analyzer=lambda tokens: tokens, # disable, only count-vectorize input tokens
)
bow_vectorizer_c.fit(docs_as_tokens) # already as tokens for Option C
print(f'The vocabulary size is {len(bow_vectorizer_c.vocabulary_)} tokens\n')
bow_vectorizer_c.get_feature_names_out()

The vocabulary size is 80 tokens



array(['a', 'across', 'algorithm', 'algorithmic', 'algorithms', 'and',
       'arbitrary', 'as', 'bias', 'biases', 'but', 'can', 'coded',
       'collected', 'computer', 'concerned', 'create', 'data',
       'decisions', 'describes', 'design', 'discrimination', 'due',
       'emerge', 'engine', 'errors', 'ethnicity', 'factors', 'found',
       'from', 'gender', 'group', 'have', 'impacts', 'in', 'inadvertent',
       'including', 'is', 'limited', 'many', 'media', 'most', 'not', 'of',
       'one', 'or', 'others', 'outcomes', 'over', 'platforms', 'privacy',
       'privileging', 'race', 'ranging', 'reflect', 'reinforcing',
       'relating', 'repeatable', 'results', 'search', 'selected',
       'sexuality', 'social', 'study', 'such', 'system', 'systematic',
       'that', 'the', 'to', 'train', 'unanticipated', 'unfair',
       'unintended', 'use', 'used', 'users', 'violations', 'way', 'with'],
      dtype=object)

## Compressed Sparse Row Matrix
The output of `transform` is a space-saving format called Compressed Sparse Row ([CSR](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html)) Matrix from SciPy.

In [13]:
#| code-summary: "Vectorize documents into count vectors in CSR matrix format"
#| code-fold: false
X_bow = bow_vectorizer_b.transform(docs)
print(type(X_bow))
print_sparsity(X_bow)

<class 'scipy.sparse._csr.csr_matrix'>
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 102 stored elements and shape (4, 80)> ... and sparsity 68.12%.


Using its `.toarray()` method, the CSR matrix can be converted to a full NumPy ndarray and then wrapped in a Pandas `DataFrame` with:

- the **documents** in the rows (or index)
- the tokens/**terms**/features in the columns
  
... hence the term **document-term matrix**. <span style="color:red">Of course, this becomes very large for larger datasets</span> so keep the data in a CSER matrix in your working NLP pipeline.

In [14]:
#| code-summary: "Display count vector as a DataFrame"
#| code-fold: false
print(f'The vocabulary size is {len(bow_vectorizer_b.get_feature_names_out())} tokens\n')
pd.DataFrame(X_bow.toarray(), index=docs, columns=bow_vectorizer_b.get_feature_names_out())

The vocabulary size is 80 tokens



,a,across,algorithm,algorithmic,algorithms,and,arbitrary,as,bias,biases,but,can,coded,collected,computer,concerned,create,data,decisions,describes,design,discrimination,due,emerge,engine,errors,ethnicity,factors,found,from,gender,group,have,impacts,in,inadvertent,including,is,limited,many,media,most,not,of,one,or,others,outcomes,over,platforms,privacy,privileging,race,ranging,reflect,reinforcing,relating,repeatable,results,search,selected,sexuality,social,study,such,system,systematic,that,the,to,train,unanticipated,unfair,unintended,use,used,users,violations,way,with
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",1,0,0,1,0,1,1,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,1,0,0,0,1,0,0,0
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0,0,2,0,0,0,0,0,1,0,1,1,1,1,0,0,0,1,1,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,1,0,0,1,1,0,4,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,5,4,1,1,0,1,1,1,0,0,1,0
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0,1,0,1,0,3,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,1,1,0,1,1,0,1,1,1,1,0,1,0,1,1,0,0,0,0,0,2,1,0,1,1,0,1,0,0,1,1,0,1,2,0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,0,0
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,1


## Binary vs, Counts
The `binary` parameter replaces token counts with a binary `1` (presence) or `0` (absence) in the vector. This is useful if the downstream task relies more on a term appearing at all rather than the number of appearances.

In [15]:
#| code-summary: "Vectorize documents into binary vectors"
#| code-fold: false
bin_vectorizer = CountVectorizer(
    tokenizer = spacy_tokenizer,
    binary = True
)
X_bin = bin_vectorizer.fit_transform(docs)
print(f'The vocabulary size is {len(bin_vectorizer.vocabulary_)} tokens\n')
print_sparsity(X_bin)
pd.DataFrame(X_bin.toarray(), index=docs, columns=bin_vectorizer.get_feature_names_out())

The vocabulary size is 80 tokens

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 102 stored elements and shape (4, 80)> ... and sparsity 68.12%.


c:\dev\hertie\Sem 5\NLP\sem5-nlp-labs\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,a,across,algorithm,algorithmic,algorithms,and,arbitrary,as,bias,biases,but,can,coded,collected,computer,concerned,create,data,decisions,describes,design,discrimination,due,emerge,engine,errors,ethnicity,factors,found,from,gender,group,have,impacts,in,inadvertent,including,is,limited,many,media,most,not,of,one,or,others,outcomes,over,platforms,privacy,privileging,race,ranging,reflect,reinforcing,relating,repeatable,results,search,selected,sexuality,social,study,such,system,systematic,that,the,to,train,unanticipated,unfair,unintended,use,used,users,violations,way,with
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",1,0,0,1,0,1,1,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,1,0,0,0,1,0,0,0
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0,0,1,0,0,0,0,0,1,0,1,1,1,1,0,0,0,1,1,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,1,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,1,1,1,0,1,1,1,0,0,1,0
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0,1,0,1,0,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,1,1,0,1,1,0,1,1,1,1,0,1,0,1,1,0,0,0,0,0,1,1,0,1,1,0,1,0,0,1,1,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,1


## Corpus-Specific Normalization
CountVectorizer also provides some **corpus-specific** normalization parameters that can be useful in controlling the number of features in your vector representation which can affect your downstream task. They are:

- `max_df`: remove common terms by maximum document frequency [0.0, 1.0] (or absolute count)
- `min_df`: remove very rare terms by minimum absolute count integer (or document frequency)
- `max_features`: limit vocabulary size, keeping the top terms by corpus frequency

`max_df` can be used for **removing corpus-specific stop words** but this does not guarantee benefit to your downstream task. We should always empirically justify their removal!

In [16]:
#| code-summary: "MODIFY: Use corpus-specific normalization"
#| code-fold: false
bow_vectorizer_norm = CountVectorizer(
    tokenizer = spacy_tokenizer,
    max_df = 1.0, # default 1,0 (100%) #MODIFY
    min_df = 1, # default 1 #MODIFY
    max_features = None #default None #MODIFY
)
X_norm = bow_vectorizer_norm.fit_transform(docs)
print(f'The vocabulary size is {len(bow_vectorizer_norm.get_feature_names_out())} tokens\n')
print_sparsity(X_norm)
pd.DataFrame(X_norm.toarray(), index=docs, columns=bow_vectorizer_norm.get_feature_names_out())

The vocabulary size is 80 tokens

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 102 stored elements and shape (4, 80)> ... and sparsity 68.12%.


c:\dev\hertie\Sem 5\NLP\sem5-nlp-labs\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,a,across,algorithm,algorithmic,algorithms,and,arbitrary,as,bias,biases,but,can,coded,collected,computer,concerned,create,data,decisions,describes,design,discrimination,due,emerge,engine,errors,ethnicity,factors,found,from,gender,group,have,impacts,in,inadvertent,including,is,limited,many,media,most,not,of,one,or,others,outcomes,over,platforms,privacy,privileging,race,ranging,reflect,reinforcing,relating,repeatable,results,search,selected,sexuality,social,study,such,system,systematic,that,the,to,train,unanticipated,unfair,unintended,use,used,users,violations,way,with
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",1,0,0,1,0,1,1,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,1,0,0,0,1,0,0,0
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0,0,2,0,0,0,0,0,1,0,1,1,1,1,0,0,0,1,1,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,1,0,0,1,1,0,4,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,5,4,1,1,0,1,1,1,0,0,1,0
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0,1,0,1,0,3,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,1,1,0,1,1,0,1,1,1,1,0,1,0,1,1,0,0,0,0,0,2,1,0,1,1,0,1,0,0,1,1,0,1,2,0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,0,0
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,1


# Bag of N-Grams Vectorization

The `ngram_range` parameter is used to tokenize by N-grams, i.e., contiguous sequences of N neighbouring tokens. This results in an increasing vocabulary size (and sparsity) as more types of N-grams are added. For example, this could be useful for sentiment analysis to keep negation 2-grams such as "does not" and "not good".

`max_features` can be used to control the vocabulary size.

In [17]:
#| code-summary: "MODIFY: Add N-grams as tokens and then vectorize documents into count vectors"
#| code-fold: false
ngram_vectorizer = CountVectorizer(
    tokenizer = spacy_tokenizer,
    ngram_range = (1, 2), # default (1, 1), unigrams only # MODIFY
    max_features = None
)
X_ngram = ngram_vectorizer.fit_transform(docs)
print(f'The vocabulary size is {len(ngram_vectorizer.vocabulary_)} tokens\n')
print_sparsity(X_ngram)
pd.DataFrame(X_ngram.toarray(), index=docs, columns=ngram_vectorizer.get_feature_names_out())

The vocabulary size is 184 tokens

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 214 stored elements and shape (4, 184)> ... and sparsity 70.92%.


c:\dev\hertie\Sem 5\NLP\sem5-nlp-labs\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,a,a computer,across,across platforms,algorithm,algorithm or,algorithmic,algorithmic bias,algorithms,algorithms that,and,and can,and ethnicity,and repeatable,and social,and unfair,arbitrary,arbitrary group,as,as privileging,bias,bias can,bias describes,bias is,biases,biases of,but,but not,can,can emerge,can have,coded,coded collected,collected,collected selected,computer,computer system,concerned,concerned with,create,create unfair,data,data is,decisions,decisions relating,describes,describes systematic,design,design of,discrimination,due,due to,emerge,emerge due,engine,engine results,errors,errors in,ethnicity,factors,factors including,found,found across,from,from inadvertent,gender,gender sexuality,group,group of,have,have impacts,impacts,impacts ranging,in,in a,inadvertent,inadvertent privacy,including,including but,is,is coded,is found,is most,limited,limited to,many,many factors,media,media platforms,most,most concerned,not,not limited,of,of algorithmic,of race,of the,of users,one,one arbitrary,or,or decisions,or the,or unanticipated,or used,others,outcomes,outcomes such,over,over others,platforms,platforms and,platforms including,privacy,privacy violations,privileging,privileging one,race,race gender,ranging,ranging from,reflect,reflect systematic,reinforcing,reinforcing social,relating,relating to,repeatable,repeatable errors,results,results and,search,search engine,selected,selected or,sexuality,sexuality and,social,social biases,social media,study,study of,such,such as,system,system that,systematic,systematic and,that,that create,that reflect,the,the algorithm,the design,the study,the unintended,the way,to,to many,to reinforcing,to search,to the,to train,train,train the,unanticipated,unanticipated use,unfair,unfair discrimination,unfair outcomes,unintended,unintended or,use,use or,used,used to,users,users over,violations,violations to,way,way data,with,with algorithms
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",1,1,0,0,0,0,1,1,0,0,1,0,0,1,0,0,1,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,1,1,0,1,1,1,1,0,0,0,0,0,0,1,1,1,1,0,0,1,1,0,1,1,1,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,1,1,1,1,0,0,0,0,1,1,1,0,0,1,0,0,0,4,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,2,1,0,1,1,4,1,0,0,2,1,1,1,1,1,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,0,0
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0,0,1,1,0,0,1,1,0,0,3,1,1,0,1,0,0,0,0,0,1,0,0,1,1,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,1,1,1,1,1,0,0,1,1,1,1,0,0,1,1,1,1,1,0,1,0,1,1,0,0,1,1,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,1,1,1,0,0,1,1,1,1,0,0,1,1,0,0,0,0,1,1,1,1,0,0,1,1,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,

# TF-IDF Vectorization

Based on the idea that the "rarity" of a word at the corpus level indicates useful information about a word's presence in a document, we now move to the idea of **Term Frequency - Inverse Document Frequency** as a calculated weight in the **document-term matrix** (yes, same shape as Bag of Words vectors, all else equal).

## TF-IDF Math

The general idea is multiplying two values for each token/term:

1. **Term Frequency** $TF(t,d)$: how often does the term appear in THIS document?
2. **Inverse Document Frequency** $IDF$: how rare is the term relative to the entire corpus?
    - $N_d$: total number of documents
    - $DF(t)$: number of documents the term appears in
    - "Inverse" because we use $\frac{N_d}{DF(t)}$ conceptually (but see below)

This results in:
- local (document) importance from $TF(t,d)$
- corpus common words weighted DOWN because of high $DF(t)$
- corpus rare words weighted UP because of low $DF(t)$

The DEFAULT implementation in scikit-learn uses **$IDF=\ln\left(\frac{1+N_d}{1+DF(t)}\right)+1$**, and also scales by dividing by the vector L2 norm $\|\mathbf{A}\|_2 = \sqrt{\sum_{i=1}^{n} A_i^2}$ (see below)

- More about [Tf-idf term weighting](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf)
- Most about the exact formula used by scikit-learn at [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer)


In [18]:
#| code-summary: "Define function for demonstrating TF-IDF calculation"
def print_math_tfidf(doc, corpus, smooth_idf=True):
    '''Demonstrate calculation of scikit-learn's TF-IDF default weight by building DataFrame'''
    # Get df(t) for each token in doc
    terms = sorted(set(doc))
    doc_freq = {
        t: sum(t in doc for doc in corpus)
        for t in terms
    }
    # Calculate each term of scikit-learn TF-IDF (default) formula
    df = pd.DataFrame(index=terms)
    df['TF(t,d)'] = [doc.count(term) for term in df.index]
    df['Nd'] = len(corpus)
    df['DF(t)'] = [doc_freq[term] for term in df.index]
    if smooth_idf:
        df['IDF = ln( (1+Nd) / (1+DF(t)) ) + 1'] = np.log((1+df['Nd'])/(1+df['DF(t)'])) + 1
        df['TF*IDF'] = df['TF(t,d)'] * df['IDF = ln( (1+Nd) / (1+DF(t)) ) + 1']
    else:
        df['IDF = ln( Nd / DF(t) ) + 1'] = np.log(df['Nd']/df['DF(t)']) + 1
        df['TF*IDF'] = df['TF(t,d)'] * df['IDF = ln( Nd / DF(t) ) + 1']
    df['L2 norm'] = np.linalg.norm(df['TF*IDF'])
    df['TF*IDF / L2'] = df['TF*IDF'] / df['L2 norm']
    return df

In [19]:
#| code-summary: "MODIFY: document index to view each term's TF-IDF calculation"
#| code-fold: false
print_math_tfidf(docs_as_tokens[2], docs_as_tokens)

,"TF(t,d)",Nd,DF(t),IDF = ln( (1+Nd) / (1+DF(t)) ) + 1,TF*IDF,L2 norm,TF*IDF / L2
across,1,4,1,1.916291,1.916291,11.753673,0.163038
algorithmic,1,4,3,1.223144,1.223144,11.753673,0.104065
and,3,4,3,1.223144,3.669431,11.753673,0.312194
bias,1,4,4,1.000000,1.000000,11.753673,0.085080
biases,1,4,1,1.916291,1.916291,11.753673,0.163038
but,1,4,2,1.510826,1.510826,11.753673,0.128541
can,1,4,2,1.510826,1.510826,11.753673,0.128541
engine,1,4,1,1.916291,1.916291,11.753673,0.163038
ethnicity,1,4,1,1.916291,1.916291,11.753673,0.163038
found,1,4,1,1.916291,1.916291,11.753673,0.163038


## TfidfVectorizer
TF-IDF is implemented in scikit-learn with the [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) class, which:

- Includes the SAME tokenization/normalization options as CountVectorizer
- Uses the SAME steps as CountVectorizer
- ⚠️ The default calculation is slightly different than the textbook (and session slides) versions as discussed above.

In [20]:
#| code-summary: "Vectorize documents into TF-IDF vectors"
#| code-fold: false
tfidf_vectorizer = TfidfVectorizer(
    tokenizer = spacy_tokenizer,
    use_idf = True, # default True / False enables TF only (IDF=1)
    smooth_idf = True, # default True (include ones in IDF log numerator and denominator)
    norm = 'l2' # default 'l2' normalization: divide by "Euclidean" length, saves one step in cosine_similarity (2nd power)
)
X_tfidf = tfidf_vectorizer.fit_transform(docs)
print(f'The vocabulary size is {len(tfidf_vectorizer.vocabulary_)} tokens\n')
print_sparsity(X_tfidf)
df_tfidf = pd.DataFrame(X_tfidf.toarray(), index=docs, columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf

The vocabulary size is 80 tokens

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 102 stored elements and shape (4, 80)> ... and sparsity 68.12%.


c:\dev\hertie\Sem 5\NLP\sem5-nlp-labs\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,a,across,algorithm,algorithmic,algorithms,and,arbitrary,as,bias,biases,but,can,coded,collected,computer,concerned,create,data,decisions,describes,design,discrimination,due,emerge,engine,errors,ethnicity,factors,found,from,gender,group,have,impacts,in,inadvertent,including,is,limited,many,media,most,not,of,one,or,others,outcomes,over,platforms,privacy,privileging,race,ranging,reflect,reinforcing,relating,repeatable,results,search,selected,sexuality,social,study,such,system,systematic,that,the,to,train,unanticipated,unfair,unintended,use,used,users,violations,way,with
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",0.217062,0.000000,0.000000,0.138548,0.000000,0.138548,0.217062,0.217062,0.113272,0.000000,0.000000,0.000000,0.000000,0.000000,0.217062,0.000000,0.217062,0.000000,0.000000,0.217062,0.000000,0.000000,0.000000,0.000000,0.000000,0.217062,0.000000,0.000000,0.000000,0.000000,0.000000,0.217062,0.000000,0.000000,0.217062,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.113272,0.217062,0.000000,0.217062,0.217062,0.217062,0.000000,0.000000,0.217062,0.000000,0.000000,0.000000,0.000000,0.000000,0.217062,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.217062,0.217062,0.171134,0.171134,0.000000,0.000000,0.000000,0.000000,0.171134,0.000000,0.000000,0.000000,0.217062,0.000000,0.000000,0.000000
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0.000000,0.000000,0.245172,0.000000,0.000000,0.000000,0.000000,0.000000,0.063971,0.000000,0.096648,0.096648,0.122586,0.122586,0.000000,0.000000,0.000000,0.122586,0.122586,0.000000,0.122586,0.000000,0.122586,0.122586,0.000000,0.000000,0.000000,0.122586,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.096648,0.078245,0.096648,0.122586,0.000000,0.000000,0.096648,0.063971,0.000000,0.490344,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.122586,0.000000,0.000000,0.000000,0.122586,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.483241,0.386593,0.122586,0.122586,0.000000,0.122586,0.122586,0.122586,0.000000,0.000000,0.122586,0.000000
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0.000000,0.163038,0.000000,0.104065,0.000000,0.312194,0.000000,0.000000,0.085080,0.163038,0.128541,0.128541,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.163038,0.000000,0.163038,0.000000,0.163038,0.163038,0.163038,0.000000,0.163038,0.163038,0.000000,0.163038,0.128541,0.104065,0.128541,0.000000,0.163038,0.000000,0.128541,0.085080,0.000000,0.000000,0.000000,0.000000,0.000000,0.326075,0.163038,0.000000,0.163038,0.163038,0.000000,0.163038,0.000000,0.000000,0.163038,0.163038,0.000000,0.163038,0.326075,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.257081,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.163038,0.000000,0.000000
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0.000000,0.000000,0.000000,0.190273,0.298099,0.190273,0.000000,0.000000,0.155561,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.298099,0.000000,0.000000,0.000000,0.000000,0.000000,0.298099,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.190273,0.000000,0.000000,0.000000,0.298099,0.000000,0.155561,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000

# Tiny Search Engine
Probably the most prominent historical uses of TF-IDF is for **document search engines**. These are based on the cosine similarity between two vectors. We can use scikit-learn's [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) method to calculate document-pair similarities quickly.

In [21]:
#| code-summary: "Display document-to-document cosine similarities"
pd.DataFrame(
    cosine_similarity(df_tfidf),
    index=docs,
    columns=docs
)

,"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.","Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.","Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.","The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination."
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",1.000000,0.014492,0.076946,0.208628
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0.014492,1.000000,0.180530,0.148364
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0.076946,0.180530,1.000000,0.125474
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0.208628,0.148364,0.125474,1.000000


## Cosine Similarity Math
The **cosine similarity** is one of the primary measures of the directional similarity of any two vectors in N-dimensional space which will be needed to enable our search engine. For any two vectors $A$ and $B$, the calculation consists of:

1. **Dot Product of Pairwise Vector Positions** in the numerator: $\mathbf{A} \cdot \mathbf{B} = \sum_{i=1}^{n} A_i B_i$
2. **Euclidean (L2) norm lengths** in the denominator for scaling: 
$\|\mathbf{A}\|_2 \|\mathbf{B}\|_2 = \sqrt{\sum_{i=1}^{n} A_i^2}\sqrt{\sum_{i=1}^{n} B_i^2}$

This results in two text documents that:

- share many terms having a cosine similarity closer to 1
- share no or few terms having a cosine similarity closer to 0

In [22]:
#| code-summary: "Define function for demonstrating cosine similarity calculation"
def print_math_cosine_similarity(doca, docb, order=False, top_k=10):
    '''docstring'''
    df = pd.DataFrame(index=doca.index)
    df['Ai'] = doca[df.index]
    df['Bi'] = docb[df.index]
    df['Ai * Bi'] = df['Ai'] * df['Bi']
    a_dot_b = df['Ai * Bi'].sum()
    l2_a = np.linalg.norm(df['Ai'])
    l2_b = np.linalg.norm(df['Bi'])
    print(f'SUM(Ai*Bi) = {a_dot_b:.5f}')
    print(f'The L2 norm length of A is ||A|| = {l2_a:.5f}')
    print(f'The L2 norm length of B is ||B|| = {l2_b:.5f}')
    print(f'Cosine similiarity is SUM(Ai*Bi) / (||A||*||B||) : {a_dot_b/(l2_a*l2_b):.5f}')
    if order:
        display(df.sort_values('Ai * Bi', ascending=False).head(top_k))
    else:
        display(df)

In [23]:
#| code-summary: "CHECK: Display cosine similarity calculation for different pairs of document indices"
#| code-fold: false
print_math_cosine_similarity(df_tfidf.iloc[0], df_tfidf.iloc[3], order=True, top_k=10)

SUM(Ai*Bi) = 0.20863
The L2 norm length of A is ||A|| = 1.00000
The L2 norm length of B is ||B|| = 1.00000
Cosine similiarity is SUM(Ai*Bi) / (||A||*||B||) : 0.20863


,Ai,Bi,Ai * Bi
that,0.171134,0.235025,0.040221
systematic,0.171134,0.235025,0.040221
unfair,0.171134,0.235025,0.040221
algorithmic,0.138548,0.190273,0.026362
and,0.138548,0.190273,0.026362
of,0.113272,0.155561,0.017621
bias,0.113272,0.155561,0.017621
across,0.000000,0.000000,0.000000
as,0.217062,0.000000,0.000000
biases,0.000000,0.000000,0.000000


## Query Relevance Search
Since any two documents in the corpus can have their similarities calculated, it follows that a **query document**, i.e., the search engine search string, works by having its **cosine similiarity** with all other documents in the corpus calculated and then ranked for relevance.

STEPS:

1. Test out different query strings and see how the toy corpus documents **rank in relevance**.
2. Check for per-term dot products to see which shared **terms contribute most** to the combined cosine similarity.

Code reference: [NLP Demystified](https://colab.research.google.com/github/futuremojo/nlp-demystified/blob/main/notebooks/nlpdemystified_vectorization.ipynb#scrollTo=jEfdfkmpP8Tv)

In [24]:
#| code-summary: "TEST: Query search string and check scores"
#| code-fold: false
query_doc = ['What is algorithmic bias?'] # MODIFY HERE

# Vectorize query
query_tfidf = tfidf_vectorizer.transform(query_doc)
df_query = pd.DataFrame(query_tfidf.toarray(), index=query_doc, columns=tfidf_vectorizer.get_feature_names_out())

# Calculate and order document relevance to query
query_similarities = pd.DataFrame(cosine_similarity(X_tfidf, query_tfidf).flatten(), index=docs, columns=query_doc)
query_similarities.sort_values(query_doc, ascending=False)

,What is algorithmic bias?
"The study of algorithmic bias is most concerned with algorithms that reflect ""systematic and unfair"" discrimination.",0.310816
"Algorithmic bias is found across platforms, including but not limited to search engine results and social media platforms, and can have impacts ranging from inadvertent privacy violations to reinforcing social biases of race, gender, sexuality, and ethnicity.",0.169993
"Algorithmic bias describes systematic and repeatable errors in a computer system that create unfair outcomes, such as privileging one arbitrary group of users over others.",0.141507
"Bias can emerge due to many factors, including but not limited to the design of the algorithm or the unintended or unanticipated use or decisions relating to the way data is coded, collected, selected or used to train the algorithm.",0.079916


In [25]:
#| code-summary: "CHECK terms contributing to similarity score"
#| code-fold: false
# MODIFY the document index numbers, also `top_k`
print_math_cosine_similarity(df_query.iloc[0], df_tfidf.loc[docs[3]], order=True, top_k=10)

SUM(Ai*Bi) = 0.31082
The L2 norm length of A is ||A|| = 1.00000
The L2 norm length of B is ||B|| = 1.00000
Cosine similiarity is SUM(Ai*Bi) / (||A||*||B||) : 0.31082


,Ai,Bi,Ai * Bi
algorithmic,0.612172,0.190273,0.116480
is,0.612172,0.190273,0.116480
bias,0.500491,0.155561,0.077857
across,0.000000,0.000000,0.000000
algorithms,0.000000,0.298099,0.000000
and,0.000000,0.190273,0.000000
arbitrary,0.000000,0.000000,0.000000
a,0.000000,0.000000,0.000000
as,0.000000,0.000000,0.000000
biases,0.000000,0.000000,0.000000


# PDF Search Engine

Let's scale up our TF-IDF-based search engine for real-life documents, specifically ten Hertie School course syllabi downloaded from
the Hertie MyStudies [Course Directory](https://mystudies.hertie-school.org/course-directory.php) filtered for "Fall 2026" (requires login).

You may try searching in any other collection of PDFs by simply pointing to another accessible folder of PDF files or copying them to the current expected folder name and location.

In [ ]:
#| code-summary: "Defne function for extracting text from a folder of PDF files"
def extract_pdf_text(folder_path):
    '''docstring'''
    folder = Path(folder_path)
    docs = []
    pdf_files = []
    for pdf_file in folder.glob('*.pdf'):
        pdf_files.append(pdf_file.name.removesuffix('.pdf'))
        doc = pymupdf.open(pdf_file)
        text = '\n'.join(page.get_text() for page in doc)
        docs.append(text)
        doc.close()
    return pd.DataFrame({'filename': pdf_files, 'text': docs})

In [ ]:
#| code-summary: "Run function on folder of PDF files"
#| code-fold: false
df_pdf_text = extract_pdf_text('pdf_corpus') # MODIFY folder location if needed
df_pdf_text.shape

## Configuration Options
You can try a few options found in this lab to improve your search engine's (subjective of course) performance:

- Use different types of document-term matrix representations: **Bag-of-Words, N-grams, TF-IDF.**
- Apply different tokenization and **normalization** using spaCy and/or scikit-learn `*Vectorizer`.
- Apply **corpus-specific** normaiization using scikit-learn `*Vectorizer`.
- Add **specific stop words** using scikit-learn `*Vectorizer`

In [ ]:
#| code-summary: "CONFIGURE: Try different tokenization, normalization, and vectorization options"
#| code-fold: false
def spacy_tokenizer(text):
    doc = nlp(text) 
    # MODIFY BELOW: lower_, is_alpha, lemma_, is_stop (normalization from Lab 2)
    tokens = [token.lower_ for token in doc if token.is_alpha]
    return tokens

# MODIFY: type of vectorizer, binary, N-gram, stop words
pdf_vectorizer = TfidfVectorizer( # OR CountVectorizer
    tokenizer = spacy_tokenizer,
    binary = False,
    ngram_range = (1, 1),
    max_df = 1.0,
    min_df = 1,
    max_features = None,
    stop_words = [] # Can input a custom list or 'english'
)

X_pdf = pdf_vectorizer.fit_transform(df_pdf_text['text'])
print(f'The vocabulary size is {len(pdf_vectorizer.vocabulary_)} tokens\n')
print_sparsity(X_pdf)
df_pdf_vec = pd.DataFrame(X_pdf.toarray(), index=df_pdf_text['filename'], columns=pdf_vectorizer.get_feature_names_out())
df_pdf_vec

## Query Relevance Search

In [ ]:
#| code-summary: "TEST different query strings on your PDF search engine"
#| code-fold: false
query_doc = ['natural language processing'] # MODIFY and check scores

# Vectorize query
query_vec = pdf_vectorizer.transform(query_doc)
df_query = pd.DataFrame(query_vec.toarray(), index=query_doc, columns=pdf_vectorizer.get_feature_names_out())

# Calculate and order document relevance to query
query_similarities = pd.DataFrame(cosine_similarity(X_pdf, query_vec).flatten(), index=df_pdf_text['filename'], columns=query_doc)
query_similarities.sort_values(query_doc, ascending=False)

In [ ]:
#| code-summary: "CHECK terms contributing to similarity score"
#| code-fold: false
# MODIFY the file name as index into the vectorized DataFrame, also `top_k`
print_math_cosine_similarity(df_query.iloc[0], df_pdf_vec.loc['E1282_NaturalLangProcess_RamirezRuiz'], order=True, top_k=10)

# Bonus!: Zipf's Law (and why we log the IDF)

How do word frequencies distribute by word ranking? Do you think that the few very common words are much more common than the many many rare words? As it turns out, there is an very interesting law for this called **Zipf's Law**:

$$
f(r) \propto \frac{1}{r^s}
$$

... where $r$ is word frequency rank and $s$ is close to 1. In other words, the second most frequent word occurs ~1/2 as often the third ~1/3, and so on. This results in a roughly straight line log-log plot of rank vs. frequency.

In [ ]:
#| code-summary: "Define function to plot rank vs. frequency (Zipf's Law)"
def plot_zipf(docs, tokenizer, log_plot=True):
    '''Plot Zipf's Law'''

    # Tokenize/normalize, combine all tokens
    docs_as_tokens = [spacy_tokenizer(doc) for doc in df_pdf_text['text']]
    all_tokens = list(chain.from_iterable(docs_as_tokens))
    len(all_tokens)
    
    # Count and rank
    freq = Counter(all_tokens)
    dfz = (
        pd.DataFrame(freq.items(), columns=['term', 'frequency'])
          .sort_values('frequency', ascending=False)
          .reset_index(drop=True)
    )
    dfz['rank'] = dfz.index + 1 #sorted
    
    # Plot
    fig = px.scatter(
        dfz,
        x='rank', y='frequency',
        text='term', # annotate terms
        hover_name='term',
        log_x=log_plot, log_y=log_plot,
        labels={'rank': 'Rank',
                'frequency': 'Frequency'
               },
        title="Zipf's Law for PDF corpus",
        width=800,
        height=800
    )
    fig.update_traces(textposition="middle right")
    fig.show()

In [ ]:
#| code-summary: "Plot Zipf's law as normal or log-log plot"
#| code-fold: false
plot_zipf(df_pdf_text['text'], spacy_tokenizer, log_plot=True)